# 과제 10 · 03 · 임베딩을 녹화해 두고 키 없이 다시 쓰기

## 현업 시나리오
의미 검색에는 임베딩 API 가 필요합니다. 그런데 학습자는 키가 없을 수 있고, 키가 있어도 같은 매뉴얼을 매번 다시 임베딩하면 돈과 시간이 듭니다. 가짜 임베딩으로 대신하면 검색 순위가 의미 없어집니다. 실제 임베딩을 **한 번 녹화해 두고 재생**하면 둘 다 피할 수 있습니다.

## 학습 목표
- 임베딩이 문장을 정해진 길이의 숫자 목록으로 바꾸는 것을 본다.
- FAISS 벡터 저장소에 넣고 찾는다.
- `CacheBackedEmbeddings` 로 녹화하고, 두 번째부터 API 를 부르지 않는 것을 확인한다.
- 녹화에 없는 문장은 가짜로 넘어가지 않고 멈추게 한다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · 문장을 vector 로

In [ ]:
# 실제 임베딩 API 대신 부를 때마다 횟수를 세는 대역을 둡니다. 뜻은 담지 않습니다.
# 같은 문장은 언제나 같은 vector 가 됩니다(문장의 sha256 으로 만듭니다).
# 관찰 포인트: 차원(size)이 vector 의 길이입니다. 앱은 256 차원으로 녹화했습니다.
import hashlib
import math
from langchain_core.embeddings import Embeddings

class PracticeEmbeddings(Embeddings):
    def __init__(self, size=256):
        self.size, self.calls = size, 0
    def _vector(self, text):
        digest = hashlib.sha256(text.encode("utf-8")).digest()
        raw = [digest[i % 32] - 127.5 for i in range(self.size)]
        norm = math.sqrt(sum(v * v for v in raw))
        return [v / norm for v in raw]
    def embed_documents(self, texts):
        self.calls += len(texts)
        return [self._vector(t) for t in texts]
    def embed_query(self, text):
        self.calls += 1
        return self._vector(text)

# 절 단위로 자르고 머리말 경로를 붙인 매뉴얼 조각입니다(Notebook 02 의 결과).
PRACTICE_SECTIONS = [
    ("MC01-MM 4.1.2", "[정비 매뉴얼 > 4.1 공구 마모 고장 (TWF) > 4.1.2 판정 기준] 공구 마모 시간이 200분 이상이면 위험 구간이다. 240분을 넘기면 사용 한계다."),
    ("MC01-MM 4.1.3", "[정비 매뉴얼 > 4.1 공구 마모 고장 (TWF) > 4.1.3 점검 절차] 날 끝 마모 폭을 측정한다. 마모 폭이 0.3mm 를 넘으면 공구(PN-TL-1008)를 교체한다."),
    ("MC01-MM 4.2.2", "[정비 매뉴얼 > 4.2 열 방산 고장 (HDF) > 4.2.2 판정 기준] 공정 온도와 공기 온도의 차이가 8.6 K 미만이고 회전수가 1,380 rpm 미만이면 해당한다."),
    ("MC01-MM 4.2.3", "[정비 매뉴얼 > 4.2 열 방산 고장 (HDF) > 4.2.3 점검 절차] 흡기 필터(PN-FL-3120)의 압력차 표시를 확인한다. 냉각팬 회전을 확인한다. 방열판 먼지를 제거한다."),
    ("MC01-MM 4.2.4", "[정비 매뉴얼 > 4.2 열 방산 고장 (HDF) > 4.2.4 권장 조치] 필터 압력차가 기준을 넘으면 흡기 필터를 교체한다. 냉각팬이 멈췄으면 냉각팬 모듈(PN-CF-3105)을 교체한다."),
    ("SOP-EA-01 8", "[이상 이벤트 대응 절차 > 8. 오탐 처리] 판정 기준에 해당하지 않는 경고는 육안·청음 점검 후 이상이 없으면 오탐 점검 기록을 남긴다."),
]

practice_embeddings = PracticeEmbeddings(size=256)
practice_vector = practice_embeddings.embed_query(PRACTICE_SECTIONS[3][1])
print("길이:", len(practice_vector), "· 앞 3개:", [round(v, 3) for v in practice_vector[:3]])
print("같은 문장은 같은 vector:", practice_vector == practice_embeddings.embed_query(PRACTICE_SECTIONS[3][1]))
print("1536 차원이면 길이:", len(PracticeEmbeddings(size=1536).embed_query("흡기 필터")))

**결과 해설** — 문장 하나가 256개 숫자가 되고, 같은 문장은 같은 vector 가 됩니다. 차원을 1536 으로 바꾸면 길이만 바뀝니다. 차원이 크면 저장 공간이 늘고, 녹화할 때와 쓸 때의 차원이 다르면 검색이 아예 되지 않습니다.

### 2단계 · FAISS 에 넣고 찾기

In [ ]:
# 조각과 vector 를 FAISS 에 넣습니다. 메타데이터에 인용 번호를 함께 둡니다.
# 찾을 때는 질의도 같은 임베딩으로 바꿔 가까운 것을 고릅니다.
# 관찰 포인트: 이 대역은 뜻을 모르므로 **여기 순위는 의미가 없습니다.** 세는 것은 API 호출 수입니다.
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

practice_documents = [Document(page_content=text, metadata={"citation": cid}) for cid, text in PRACTICE_SECTIONS]
practice_embeddings.calls = 0
practice_store = FAISS.from_documents(practice_documents, practice_embeddings)
practice_found = practice_store.similarity_search("필터 압력차 확인", k=2)
print("찾은 것:", [d.metadata["citation"] for d in practice_found], "(순위는 대본)")
print("API 호출:", practice_embeddings.calls, "번 = 조각", len(practice_documents), "+ 질의 1")

**결과 해설** — 조각 6개를 넣으며 6번, 질의 하나로 1번, 모두 7번 불렀습니다. 앱을 켤 때마다 매뉴얼 청크 69개를 다시 임베딩하면 그만큼 부르고, 키가 없으면 아예 시작하지 못합니다.

### 3단계 · 녹화해 두고 다시 쓰기

In [ ]:
# CacheBackedEmbeddings 는 처음 본 문장만 API 로 보내고 결과를 저장소에 남깁니다.
# 키는 namespace + sha256(문장) 입니다. namespace 에 모델과 차원을 넣어 섞이지 않게 합니다.
# 관찰 포인트: 두 번째에는 API 를 한 번도 부르지 않습니다. 저장소에서 꺼냅니다.
import tempfile
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

practice_folder = tempfile.mkdtemp()
practice_api = PracticeEmbeddings(size=256)
practice_recorder = CacheBackedEmbeddings.from_bytes_store(
    practice_api, LocalFileStore(practice_folder), namespace="practice-256-",
    query_embedding_cache=True, key_encoder="sha256")

practice_texts = [text for _, text in PRACTICE_SECTIONS]
practice_recorder.embed_documents(practice_texts)
practice_recorder.embed_query("필터 압력차 확인")
practice_first = practice_api.calls
practice_recorder.embed_documents(practice_texts)
practice_recorder.embed_query("필터 압력차 확인")
practice_second = practice_api.calls - practice_first
practice_keys = list(LocalFileStore(practice_folder).yield_keys())
print("처음:", practice_first, "번 호출 · 다시:", practice_second, "번 호출")
print("녹화된 파일:", len(practice_keys), "개 · 예:", practice_keys[0][:40] + "…")

**결과 해설** — 처음에는 조각 6개와 질의 1개로 7번 불렀고, 같은 것을 다시 요청하자 0번입니다. 녹화본은 파일 7개로 남습니다. 이 폴더를 저장소에 함께 올리면, 학습자는 키 없이도 **실제 임베딩의 검색 순위**를 봅니다.

### 4단계 · 녹화에 없는 문장은 멈춘다

In [ ]:
# 재생 전용으로 엽니다. 녹화에 없는 문장이 오면 API 대신 이 대역이 불립니다.
# 가짜 vector 를 만들어 넘어가지 않고 예외로 멈춥니다. 가짜 순위를 진짜처럼 내는 것보다 낫습니다.
# 관찰 포인트: 녹화된 문장은 API 없이 같은 vector 가 나오고, 새 문장은 이유를 말하며 멈춥니다.
class PracticeNotRecorded(RuntimeError):
    pass

class PracticeUnrecorded(Embeddings):
    def embed_documents(self, texts):
        raise PracticeNotRecorded(f"녹화되지 않은 문장: {texts[0][:30]!r}. live 로 돌리거나 다시 녹화하세요.")
    def embed_query(self, text):
        return self.embed_documents([text])[0]

practice_replay = CacheBackedEmbeddings.from_bytes_store(
    PracticeUnrecorded(), LocalFileStore(practice_folder), namespace="practice-256-",
    query_embedding_cache=True, key_encoder="sha256")
practice_same = practice_replay.embed_query("필터 압력차 확인") == practice_api.embed_query("필터 압력차 확인")
try:
    practice_replay.embed_query("녹화하지 않은 새 질문")
    practice_stopped = None
except PracticeNotRecorded as error:
    practice_stopped = str(error)
print("녹화된 질의 재생이 원본과 같다:", practice_same)
print("새 질의:", practice_stopped)

assert len(practice_vector) == 256
assert practice_first == len(practice_texts) + 1 and practice_second == 0, "두 번째에도 API 를 불렀습니다"
assert practice_same, "재생한 vector 가 녹화와 다릅니다"
assert practice_stopped and "녹화되지 않은" in practice_stopped, "녹화에 없는 문장이 가짜로 넘어갔습니다"

**결과 해설** — 녹화된 질의는 API 없이 원본과 같은 vector 로 재생되고, 녹화하지 않은 질의는 무엇이 없는지 말하며 멈춥니다. 앱은 이 예외를 503 으로 바꿔 "live 로 돌리거나 다시 녹화하라"고 화면에 알립니다.

## 직접 해 볼 과제
namespace 를 "practice-1536-" 으로 바꿔 재생 전용으로 열어 보세요. 같은 문장인데 왜 멈춥니까? 차원이나 모델을 바꾼 뒤 옛 녹화를 그대로 쓰면 어떤 일이 생길지 생각해 보세요.

In [ ]:
# namespace 만 바꿔 같은 폴더를 엽니다. 녹화 키가 namespace 로 시작하므로 찾지 못합니다.
# 다른 모델·차원의 vector 가 섞이지 않게 하는 장치입니다.
practice_try_namespace = "practice-1536-"
practice_try = CacheBackedEmbeddings.from_bytes_store(
    PracticeUnrecorded(), LocalFileStore(practice_folder), namespace=practice_try_namespace,
    query_embedding_cache=True, key_encoder="sha256")
try:
    print(len(practice_try.embed_query("필터 압력차 확인")))
except PracticeNotRecorded as error:
    print("멈춤:", error)

## 중간 결과
vector 길이(256), FAISS 적재 때의 API 호출 수(7), 녹화 후 다시 요청할 때의 호출 수(0), 재생 vector 가 원본과 같은지, 그리고 녹화에 없는 문장이 멈추는지를 확인합니다.

## 실패 경계
녹화본은 **녹화한 문장에만** 답합니다. 매뉴얼을 한 글자 고치거나 질의 만드는 규칙을 바꾸면 녹화를 다시 해야 합니다. 그때 녹화에 없는 문장을 가짜 vector 로 채우면 앱은 아무 경고 없이 엉뚱한 순위를 냅니다. 그래서 멈추게 했습니다.

## 실제 app 연결
과제 10 App 의 `evidence.py::recorded_embeddings` 가 같은 방식(namespace `text-embedding-3-small-256-`, sha256 키)으로 `data/index/recorded_embeddings/` 를 재생하고, `evidence.py::_Unrecorded` 가 녹화에 없는 문장을 멈춥니다. 녹화는 `scripts/task10/record_embeddings.py` 로 한 번 합니다(OpenAI 비용이 듭니다). 앱의 벡터 저장소는 Postgres 의 pgvector 이고, 적재 스크립트 `scripts/task10/ingest_manuals.py` 가 같은 녹화 임베딩으로 `shared/rag/store.py::build_store` 를 불러 한 번 넣습니다. 이 Notebook 의 FAISS 는 메모리에 잠깐 만드는 연습용입니다.

## 다음 Notebook 연결
다음 `04_hybrid_search.ipynb` 에서는 EventCard 로 질의를 만들고, 글자 검색과 의미 검색을 함께 씁니다.